## 무신사 방식 "구매 평가" 화면 만들기
### N-gram + Bag-of-Words

**환경:** Google Colab / Jupyter Notebook
**준비물:** `TS_1-2` 엑셀 파일 (남성의류 상품평 8,001건)
**소요 시간:** 약 40분

## 오늘 만들 결과
쇼핑몰에서 자주 보는 **"구매한 회원들이 평가했어요"** 화면을 리뷰 글만으로 만듭니다.

```
사이즈   매우 작아요   ▏               2%
         조금 작아요   ██              22%
         정사이즈예요  ████            38%   ← 가장 많은 답은 파란 막대
         조금 커요     ████            36%
         매우 커요     ▏                3%
```

> ⚠️ 쇼핑몰 화면은 구매자가 **직접 고른 답**(설문)을 센 것입니다. 우리 엑셀에는 그런 답이 없으므로, **리뷰 글에 나온 표현**("너무 커요", "조금 작아요")을 찾아서 답을 **추정**합니다.

## 오늘 쓰는 기술 두 가지
| 기술 | 한 줄 설명 | 오늘 하는 일 |
|---|---|---|
| **N-gram** | 붙어 있는 단어 N개를 한 묶음으로 보기 | `크다`만 보면 정도를 모름 → `너무 크다`(2-gram)로 **"매우 커요"** 구분 |
| **Bag-of-Words (BoW)** | 문서를 "단어(표현)별 개수" 표로 바꾸기 | 리뷰마다 평가 표현이 몇 번 나왔는지 세기 |

scikit-learn의 `CountVectorizer` 하나로 두 기술을 함께 씁니다. (`ngram_range` = N-gram, 결과 표 = BoW)

## 실습 진행 과정
| 단계 | 할 일 | 확인할 결과 |
|---|---|---|
| 0. 준비 | 설치, 엑셀 읽기 | 리뷰 수 |
| 1. 단어 나누기 | Kiwi로 리뷰를 단어로 나누기 | 원문과 단어 비교 |
| 2. N-gram과 BoW 알아보기 | 문장 3개로 연습 → 전체 리뷰에서 2-gram 세기 | `너무 크다`, `좀 작다` |
| 3. 표현 사전 | 항목별 5단계 답과 표현 정하기 | `ASPECTS` 사전 |
| 4. 리뷰마다 답 고르기 | BoW로 표현 세고 답 고르기 | 리뷰별 답 |
| 5. 비율 계산 | 답별 리뷰 수와 % | 항목별 비율 표 |
| 6. 화면 만들기 | HTML로 평가 화면 그리기 | 완성 화면 |
| 7. 정리 | 결과 확인, 한계 | |

> ✏️ 표시가 있는 곳은 **직접 바꿔 볼 수 있는 부분**입니다.

---
## 0. 준비


In [1]:
# 한국어 형태소 분석기와 엑셀 읽기 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 100.7 MB/s eta 0:00:00


Colab 왼쪽 **📁 파일 창**에 `TS_1-2`로 시작하는 엑셀 파일을 끌어다 놓으세요. 아래 셀이 파일 이름을 **자동으로 찾습니다.**

In [4]:
# 표 다루기 도구와 파일 찾기 도구
import pandas as pd
import glob

# ===== ✏️ [입력] 찾을 엑셀 파일 이름 ("TS_1-2.남성의류.xlsx")=====
excel_path = ("TS_1-2.남성의류.xlsx")
# =========================================================================

# 엑셀에서 상품평 열만 읽고, 빈 리뷰는 빼기
df = pd.read_excel(excel_path, usecols=["상품평"]).dropna()
df["상품평"] = df["상품평"].astype(str)
print("파일:", excel_path)
print("리뷰 수:", len(df))
df.head(3)

파일: TS_1-2.남성의류.xlsx
리뷰 수: 8001


,상품평
0,시아버님 선물용으로 사드리니 팔이 조금 길고 잘 안맞으신다고 색상도 마음에 안든다고하네요
1,품질은 딱 가격만큼만 합니다\n가볍긴 하나\n방송만큼 따뜻한 정도는 \n아니고\n자...
2,감촉도 좋고 품질 좋고 입은 모습이 저에게 아주 잘 어울려\n대단히 기쁘게 잘 착...


---
## 1. 단어 나누기
한국어는 `커요`, `커서`, `크고`처럼 모양이 계속 바뀝니다. **Kiwi**로 모두 `크다`로 바꿔야 같은 단어로 셀 수 있습니다.

| 남기는 것 | 예 | 바꾸는 방법 |
|---|---|---|
| 명사 | `사이즈`, `색상`, `품질` | 그대로 |
| 동사·형용사 | `커요` → `크다` | 뒤에 `다` 붙이기 |
| 어근 | `저렴` → `저렴하다` | 뒤에 `하다` 붙이기 |
| 정도·부정·평가 표현 | `너무`, `조금`, `안`, `않다`, `별로` | 그대로 (N-gram에서 꼭 필요) |

마지막에 단어를 **띄어쓰기로 이어 붙인 문장**을 만듭니다. scikit-learn 도구가 이 문장을 읽습니다.

In [7]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 리뷰 원문을 형태소 분석한 뒤, 필요한 단어만 골라
#                  띄어쓰기로 이어 붙인 '문장' 열을 만듦
# ------------------------------------------------------------

# Kiwi 형태소 분석기
from kiwipiepy import Kiwi
kiwi = Kiwi()

# 남길 정도·부정·평가 표현
# set(집합)으로 만들어서 'in' 검사를 빠르게 함
KEEP = {"너무", "매우", "아주", "엄청", "정말", "진짜", "완전", "많이",
        "조금", "좀", "약간", "살짝", "딱", "안", "못", "별로"}

# 형태소 분석 결과를 단어 목록으로 바꾸는 함수
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag               # 'VA' 동사 태그
        if tag in {"VV", "VA"} or t.form in {"않", "없"}:
            words.append(t.form + "다")           # 동사·형용사·'않'·'없'은 '다' 붙이기
        elif tag == "XR":
            words.append(t.form + "하다")         # 어근은 '하다' 붙이기
        elif tag in {"NNG", "NNP"} or t.form in KEEP:
            words.append(t.form)                 # 명사와 정도·부정·평가 표현은 그대로
    return words

# 리뷰 전체를 한 번에 분석하고, 단어를 띄어쓰기로 이어 붙이기 (10초 정도 걸림)
df["문장"] = [" ".join(to_words(tokens)) for tokens in kiwi.tokenize(df["상품평"].tolist())]
# df["상품평"].tolist()
#   - '상품평' 열을 파이썬 리스트(문자열 목록)로 변환
# kiwi.tokenize(문자열 목록)
#   - 문자열 목록을 넘기면 리뷰마다 분석한 토큰 목록을 입력 순서대로 하나씩 돌려줌
# to_words(tokens)
#   - 리뷰 하나의 토큰 목록에서 남길 단어 리스트를 만듦
# " ".join(리스트)
#   - 단어 리스트를 띄어쓰기 한 칸으로 이어 붙인 문자열로 만듦
#   - 예: ["사이즈", "너무", "크다"] → "사이즈 너무 크다"
# [ ... for tokens in ... ]
#   - 모든 리뷰에 대해 위 과정을 반복해 만든 문자열들의 리스트
# df["문장"] = 리스트
#   - 그 리스트를 '문장'이라는 새 열로 저장 (리스트 순서대로 행에 들어감)

# 원문과 바뀐 문장 비교
# [1, 6]: 위치 기준 2번째, 7번째 행
for i in [1, 6]:
    print("원문:", df["상품평"][i])
    print("단어:", df["문장"][i])
    print()

원문: 품질은 딱 가격만큼만 합니다
가볍긴 하나
방송만큼 따뜻한 정도는 
아니고
자켓은 간절기용으로는 
적당할 듯 합니다
단어: 품질 딱 가격 하다 방송 따뜻하다 정도 자켓 간절기 적당하다 하다

원문: 싸서 두세트 더 사놨어요
보풀나면 버리고 새로
입으려고요
색도 좋네요
단어: 싸다 세트 사다 보 풀 나다 버리다 색 좋다



---
## 2. N-gram과 Bag-of-Words 알아보기

### 2-1. 짧은 문장 3개로 연습
**Bag-of-Words(BoW)** 는 문장을 "단어별 개수" 표로 바꿉니다. 단어 순서는 보지 않고 **가방에 단어를 쏟아붓듯** 개수만 셉니다.
**N-gram**은 붙어 있는 단어 N개를 한 묶음으로 봅니다.

| `ngram_range` | 세는 것 | `사이즈 너무 크다`에서 |
|---|---|---|
| `(1, 1)` | 1-gram (단어 하나) | `사이즈`, `너무`, `크다` |
| `(2, 2)` | 2-gram (두 단어 묶음) | `사이즈 너무`, `너무 크다` |
| `(1, 2)` | 1-gram + 2-gram | 위 5개 모두 |

In [9]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 짧은 문장 3개로 1-gram Bag-of-Words 표를 만들어 보기
# ------------------------------------------------------------

# 단어(표현) 개수 세기 도구
# CountVectorizer: 문서마다 단어(또는 N-gram)가 몇 번 나오는지 세어 '문서 × 단어' 개수 행렬을 만드는 scikit-learn 도구
from sklearn.feature_extraction.text import CountVectorizer

# 띄어쓰기로만 자르고, 한 글자 단어(좀, 딱, 안)도 남기는 설정
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 연습용 문장 3개
sample = ["사이즈 너무 크다", "사이즈 조금 작다", "사이즈 크다 않다 적당하다"]

# 1-gram BoW: 단어 하나씩 세기
cv1 = CountVectorizer(ngram_range=(1, 1), **opts)
pd.DataFrame(cv1.fit_transform(sample).toarray(), columns=cv1.get_feature_names_out(), index=sample)
# cv1.fit_transform(sample)
#   - fit: sample에 나온 모든 1-gram을 모아 단어 목록(vocabulary)을 만듦
#   - transform: 문장마다 각 단어의 등장 횟수를 센 희소 행렬(sparse matrix)을 만듦
# .toarray()
#   - 희소 행렬을 일반 2차원 배열로 변환
# cv1.get_feature_names_out()
#   - 행렬의 각 열에 해당하는 단어 이름 배열 (정렬된 순서)
# pd.DataFrame(..., columns=단어 이름, index=sample)
#   - 행 = 문장, 열 = 단어, 값 = 등장 횟수 인 표로 만들어 출력

,너무,사이즈,않다,작다,적당하다,조금,크다
사이즈 너무 크다,1,1,0,0,0,0,1
사이즈 조금 작다,0,1,0,1,0,1,0
사이즈 크다 않다 적당하다,0,1,1,0,1,0,1


1-gram 표에서는 첫째 문장과 셋째 문장 모두 `크다`가 1입니다. **"너무 크다"인지 "크지 않다"인지 구분이 안 됩니다.**

In [10]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 같은 문장 3개로 2-gram Bag-of-Words 표를 만들기
# ------------------------------------------------------------
# 2-gram BoW: 두 단어 묶음 세기
cv2 = CountVectorizer(ngram_range=(2, 2), **opts)

# 2-gram 목록을 만들고(fit) 문장별 개수를 센 뒤(transform), 행 = 문장 / 열 = 2-gram 인 표로 출력
pd.DataFrame(cv2.fit_transform(sample).toarray(), columns=cv2.get_feature_names_out(), index=sample)

,너무 크다,사이즈 너무,사이즈 조금,사이즈 크다,않다 적당하다,조금 작다,크다 않다
사이즈 너무 크다,1,1,0,0,0,0,0
사이즈 조금 작다,0,0,1,0,0,1,0
사이즈 크다 않다 적당하다,0,0,0,1,1,0,1


2-gram 표에서는 `너무 크다`, `조금 작다`, `크다 않다`가 따로 보입니다. → **정도와 부정을 구분할 수 있습니다.**

### 2-2. 전체 리뷰에서 2-gram 세기
이제 리뷰 8,001건 전체에 2-gram BoW를 적용해 **어떤 정도 표현이 실제로 쓰이는지** 찾아봅니다.

In [14]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 전체 리뷰의 '문장' 열에 2-gram BoW를 적용하고,
#                  2-gram별 전체 등장 횟수를 많은 순서로 보기
# ------------------------------------------------------------
import numpy as np
# 전체 리뷰에 2-gram BoW 적용
cv_all = CountVectorizer(ngram_range=(2, 2), **opts)

# 전체 리뷰 문장에서 2-gram 목록을 만들고 리뷰별 개수를 셈
# X_all: (리뷰 수 × 2-gram 종류 수) 크기의 희소 행렬
#   - 대부분 칸이 0이라서 0이 아닌 값만 저장하는 희소 행렬 형태로 반환됨
X_all = cv_all.fit_transform(df["문장"])
print("리뷰 수, 2-gram 종류 수:", X_all.shape)

# 2-gram별 전체 개수를 더해서 많은 순서로 정렬
bigram_count = pd.Series(np.asarray(X_all.sum(axis=0)).ravel(), index=cv_all.get_feature_names_out())

# .sort_values(ascending=False): 값(등장 횟수)이 큰 순서로 정렬
# .head(15): 위에서 15개만 출력
bigram_count.sort_values(ascending=False).head(15)

리뷰 수, 2-gram 종류 수: (8001, 40736)


,0
맘 들다,712
가격 대비,593
마음 들다,313
너무 좋다,281
딱 맞다,245
좋다 가격,236
좋다 같다,233
좋다 하다,231
저렴하다 가격,229
가성비 좋다,223


In [17]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 특정 단어(target)가 포함된 2-gram만 골라 많은 순서로 보기
# ------------------------------------------------------------
# ✏️ 찾고 싶은 단어 ("작다", "따뜻하다", "밝다", "품질"로 바꿔 보기)
target = "크다"

# 그 단어가 들어간 2-gram만 골라 많은 순서로 보기
# bigram_count.index: 모든 2-gram 문자열 (예: "너무 크다")
# gram.split(): 2-gram을 공백으로 잘라 두 단어 리스트로 만듦 (예: ["너무", "크다"])
# target in gram.split(): 두 단어 중 하나가 target과 '정확히 같으면' True
#   - 문자열 부분 일치가 아니라 단어 단위 비교이므로 "크다"로 "커다"나 "크다랗다"는 잡히지 않음
# 결과: 2-gram마다 True/False 가 들어 있는 리스트 (bigram_count와 길이·순서가 같음)
has_target = [target in gram.split() for gram in bigram_count.index]


# bigram_count[has_target]
#   - True/False 리스트로 인덱싱하면 True인 위치의 항목만 남음
# 남은 2-gram을 등장 횟수가 큰 순서로 정렬해 상위 10개 출력
bigram_count[has_target].sort_values(ascending=False).head(10)

,0
크다 나오다,154
사이즈 크다,147
좀 크다,115
조금 크다,77
치수 크다,76
크다 주문,64
너무 크다,49
크다 교환,42
크다 사이즈,40
약간 크다,36


### 결과 읽기
- `엄청 크다`, `너무 크다` → **매우 커요**
- `좀 크다`, `조금 크다`, 그냥 `크다` → **조금 커요**
- `크다 않다` (= 크지 않다) → 크다는 뜻이 **아님**
- `크다 주문` (= 크게 주문했다) → 옷이 **작게 나왔다**는 뜻일 때가 많음 → **조금 작아요**

이렇게 찾은 표현으로 다음 단계에서 **표현 사전**을 만듭니다.

---
## 3. 표현 사전 만들기
2단계에서 찾은 표현으로 **항목별 5단계 답**과 **각 답에 해당하는 표현(N-gram)** 을 정합니다.

| 단계 | 사이즈 | N-gram |
|---|---|---|
| 매우 작아요 | `너무 작다`, `엄청 작다` … | 2-gram |
| 조금 작아요 | `작다` | 1-gram |
| 정사이즈예요 | `딱 맞다`, `사이즈 맞다`, `크다 않다` | 2-gram |
| 조금 커요 | `크다` | 1-gram |
| 매우 커요 | `너무 크다`, `엄청 크다` … | 2-gram |

`strong(["크다"])`는 `너무 크다`, `매우 크다`, `엄청 크다` … 를 자동으로 만들어 줍니다.

In [18]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 단어 앞에 강한 정도 표현을 붙인 2-gram 목록을 자동으로 만드는 함수 정의
# ------------------------------------------------------------

# 강한 정도 표현
STRONG = ["너무", "매우", "아주", "엄청", "정말", "진짜", "완전", "많이"]

# 단어 앞에 강한 정도 표현을 붙인 2-gram 목록을 만드는 함수
# words(단어 리스트)를 받아 'STRONG의 각 표현 + 공백 + 단어' 형태의 2-gram 리스트를 돌려주는 함수
#   - 바깥 반복: STRONG의 표현 s를 하나씩
#   - 안쪽 반복: words의 단어 w를 하나씩
#   - f"{s} {w}": 두 문자열을 공백 한 칸으로 이어 붙임
#   - 결과 개수 = len(STRONG) × len(words)
def strong(words):
    return [f"{s} {w}" for s in STRONG for w in words]

print(strong(["크다"]))

['너무 크다', '매우 크다', '아주 크다', '엄청 크다', '정말 크다', '진짜 크다', '완전 크다', '많이 크다']


In [19]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 항목별 5단계 답과, 각 답으로 판단할 표현(N-gram) 목록을 정의
# ------------------------------------------------------------

# ===== ✏️ [입력] 항목별 5단계 답과 표현 (화면에 나오는 순서대로) =====
ASPECTS = {
    "사이즈": {
        "매우 작아요":   strong(["작다", "끼다"]),
        "조금 작아요":   ["작다", "끼다", "크다 주문", "크다 시키다"],
        "정사이즈예요":  ["정사이즈", "정 사이즈", "딱 맞다", "사이즈 맞다", "싸이즈 맞다", "치수 맞다", "품 맞다",
                         "사이즈 딱", "크다 않다", "작다 않다"],
        "조금 커요":     ["크다", "넉넉하다", "작다 주문", "작다 시키다"],
        "매우 커요":     strong(["크다", "넉넉하다"]),
    },
    "화면 대비 색감": {
        "매우 밝아요":   strong(["밝다", "연하다"]),
        "약간 밝아요":   ["밝다", "연하다"],
        "화면과 비슷해요": ["화면 같다", "사진 같다", "화면 동일", "실물 같다", "색상 같다", "색 같다", "화면 똑같다", "사진 똑같다"],
        "약간 어두워요": ["어둡다", "진하다"],
        "매우 어두워요": strong(["어둡다", "진하다"]),
    },
    "퀄리티": {
        "매우 별로예요": strong(["실망"]) + ["품질 너무 별로", "품질 정말 별로"],
        "별로예요":      ["실망", "불량", "보풀", "실밥", "품질 별로", "재질 별로", "원단 별로", "저렴하다 보이다", "싸다 보이다", "품질 안 좋다"],
        "괜찮아요":      ["품질 괜찮다", "재질 괜찮다", "원단 괜찮다", "괜찮다 품질", "품질 무난하다", "품질 보통"],
        "좋아요":        ["품질 좋다", "재질 좋다", "원단 좋다", "좋다 품질", "좋다 재질", "좋다 원단", "고급",
                         "보풀 없다", "보풀 안", "불량 없다", "실밥 없다"],
        "매우 좋아요":   [f"{n} {s} 좋다" for n in ["품질", "재질", "원단"] for s in STRONG],
    },
    "보온성": {
        "전혀 따뜻하지 않아요": ["따뜻하다 않다", "안 따뜻하다", "따듯하다 않다", "안 따듯하다"],
        "조금 따뜻해요":   ["조금 따뜻하다", "좀 따뜻하다", "약간 따뜻하다", "살짝 따뜻하다"],
        "적당해요":       ["따뜻하다 적당하다", "적당하다 따뜻하다", "보온 적당하다", "적당하다 보온"],
        "꽤 따뜻해요":     ["따뜻하다", "따듯하다", "보온 좋다", "포근하다"],
        "매우 따뜻해요":   strong(["따뜻하다", "따듯하다"]),
    },
}
# ==========================================================================

# 항목별 표현 개수 확인
for name, levels in ASPECTS.items():
    print(name, {level: len(phrases) for level, phrases in levels.items()})

사이즈 {'매우 작아요': 16, '조금 작아요': 4, '정사이즈예요': 10, '조금 커요': 4, '매우 커요': 16}
화면 대비 색감 {'매우 밝아요': 16, '약간 밝아요': 2, '화면과 비슷해요': 8, '약간 어두워요': 2, '매우 어두워요': 16}
퀄리티 {'매우 별로예요': 10, '별로예요': 10, '괜찮아요': 6, '좋아요': 11, '매우 좋아요': 24}
보온성 {'전혀 따뜻하지 않아요': 4, '조금 따뜻해요': 4, '적당해요': 4, '꽤 따뜻해요': 4, '매우 따뜻해요': 16}


---
## 4. 리뷰마다 답 고르기
2단계에서는 **모든** 2-gram을 셌지만, 여기서는 `vocabulary=`로 **3단계에서 정한 표현만** 세는 BoW를 만듭니다.
`ngram_range=(1, 3)`은 1-gram부터 3-gram까지 모두 찾으라는 뜻입니다.

```
             너무 작다  작다  사이즈 적당하다  크다  너무 크다
리뷰 0          0       0         0          1      0
리뷰 1          0       0         0          1      1      ← '너무 크다'와 '크다'가 모두 1
리뷰 2          0       0         0          0      0      ← 사이즈 이야기 없음
```

### 답 고르는 규칙
1. 표현이 하나도 없으면 → 그 항목을 **말하지 않은 리뷰** (계산에서 빠짐)
2. 표현이 여러 개면 → **가장 긴 표현(N이 큰 N-gram)** 을 고름
   - 위 표의 리뷰 1은 `너무 크다`(2단어)가 `크다`(1단어)보다 길어서 **매우 커요**
3. 길이가 같으면 → 사전에서 **먼저 나온 표현**

In [21]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 한 항목의 표현 목록만 세는 BoW 표(리뷰 × 표현)를 만드는 함수 정의 및 확인
# ------------------------------------------------------------

# 한 항목의 표현을 리뷰마다 세는 함수 (정해 둔 표현만 세는 BoW)
# levels: 한 항목의 답 딕셔너리 (예: ASPECTS["사이즈"])
# 반환값: 행 = 리뷰, 열 = 표현, 값 = 그 리뷰에서 표현이 나온 횟수 인 DataFrame
def count_phrases(levels):
    phrases = [p for ps in levels.values() for p in ps]                 # 표현 목록 펼치기
    cv = CountVectorizer(vocabulary=phrases, ngram_range=(1, 3), **opts)
    X = cv.fit_transform(df["문장"])
    return pd.DataFrame(X.toarray(), columns=phrases)                    # 리뷰 × 표현 개수 표

# 사이즈 항목으로 확인 (표현이 하나라도 있는 리뷰 5개)
levels = ASPECTS["사이즈"]
bow = count_phrases(levels)
bow[bow.sum(axis=1) > 0].head()

,너무 작다,너무 끼다,매우 작다,매우 끼다,아주 작다,아주 끼다,엄청 작다,엄청 끼다,정말 작다,정말 끼다,...,엄청 크다,엄청 넉넉하다,정말 크다,정말 넉넉하다,진짜 크다,진짜 넉넉하다,완전 크다,완전 넉넉하다,많이 크다,많이 넉넉하다
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: BoW 표의 한 줄(리뷰 하나)을 보고 5단계 답 중 하나를 고르는 함수 정의 및 적용
# ------------------------------------------------------------

# row: BoW 표의 한 행 (index = 표현 이름, 값 = 등장 횟수 인 Series)
# levels: 그 항목의 답 딕셔너리
def choose_level(row, levels):
    # row.index를 순서대로 돌면서 값이 0보다 큰(리뷰에 나온) 표현만 리스트로 모음
    #   - 순서는 BoW 표의 열 순서 = 사전에 작성한 표현 순서
    found = [p for p in row.index if row[p] > 0]                          # 나온 표현들

    # 나온 표현이 하나도 없으면 None 반환 → 이 리뷰는 이 항목을 말하지 않은 것으로 처리
    if not found:
        return None                                                        # 항목을 말하지 않음

    #이 코드는 found에 있는 문자열 중 공백으로 구분한 단어 수가 가장 많은 문자열을 선택하는 코드
        # max(found, key=...)
    #   - key 함수 결과가 가장 큰 표현을 고름
    #   - len(p.split()): 표현을 공백으로 나눈 단어 개수 = N-gram의 N
    #   - 단어 개수가 같은 표현이 여러 개면, max는 found에서 먼저 나온 것을 반환
    #     → 사전에서 먼저 작성된 표현이 선택됨
    best = max(found, key=lambda p: len(p.split()))                        # 가장 긴 표현

    # levels.items()를 순서대로 돌며 best가 들어 있는 답 이름을 찾아 첫 번째 것을 반환
    #   - next(제너레이터): 조건을 만족하는 첫 번째 값만 꺼냄
    return next(level for level, ps in levels.items() if best in ps)       # 그 표현의 답

# bow.apply(choose_level, axis=1, levels=levels)
#   - axis=1: BoW 표의 각 행(리뷰)을 choose_level의 첫 번째 인자 row로 넘김
#   - levels=levels: choose_level의 levels 인자로 함께 전달
#   - 결과: 리뷰마다 고른 답(또는 None)이 들어 있는 Series (index = bow의 행 번호)
# df["사이즈"] = 결과 Series
#   - df에 '사이즈' 열로 저장 (pandas는 행 번호(index)가 같은 행끼리 값을 맞춰 넣음)
df["사이즈"] = bow.apply(choose_level, axis=1, levels=levels)

# df["사이즈"].notna(): '사이즈' 값이 None/NaN이 아닌 행만 True
# df.loc[조건, [열 목록]]: 조건이 True인 행에서 '상품평', '사이즈' 두 열만 선택
df.loc[df["사이즈"].notna(), ["상품평", "사이즈"]].head()

,상품평,사이즈
4,"아들 운동 할 때 ,\n입으라고 구입했는데 \n너무 맘에 안드네요.\n긁혀서 올이 ...",조금 커요
7,평소 100사이즈 입는데 사이즈가 작네요\n받자마자 입어보고 바로 반품신청했습니다 ...,조금 작아요
10,옷이 끼는게 싫어서 한 칫수 크게 주문했더니 맞네요.\n딱 원했던 핏 이네요.\n,조금 작아요
12,간편하게 부담스럽지않게 편하게 입을거 같습니다 사이즈도 넉넉하니 좋습니다,조금 커요
13,사이즈교환을했는데금방다시받았어요.신랑이183키에105사이즈는팔이좀작아서105사이즈를...,조금 작아요


---
## 5. 비율 계산
답별로 리뷰 수를 세고(`value_counts`), 그 항목을 말한 리뷰 수로 나누면 **비율(%)** 입니다.
`reindex`는 답을 사전 순서대로 줄 세우고, 한 번도 안 나온 답은 0으로 채웁니다.

In [25]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: '사이즈' 답별 리뷰 수와 비율(%) 표 만들기
# ------------------------------------------------------------

# df["사이즈"].value_counts()
#   - 답별 리뷰 수를 셈 (None/NaN 값은 기본 설정에 따라 세지 않음)
# .reindex(list(levels), fill_value=0)
#   - list(levels): 답 딕셔너리의 키 목록 = 사전에 작성한 5단계 답 순서
#   - 결과를 그 순서대로 다시 정렬
#   - 한 번도 나오지 않은 답은 fill_value=0으로 채움
count = df["사이즈"].value_counts().reindex(list(levels), fill_value=0)

# count.sum(): 사이즈를 말한 리뷰 수 합계 (5개 답의 리뷰 수를 모두 더한 값)
# count / count.sum() * 100: 답별 비율(%)
# .round(): 소수점 첫째 자리에서 반올림
# .astype(int): 정수로 변환
percent = (count / count.sum() * 100).round().astype(int)

# '리뷰 수', '비율(%)' 두 열을 가진 DataFrame으로 만들어 출력 (행 = 답)
pd.DataFrame({"리뷰 수": count, "비율(%)": percent})

,리뷰 수,비율(%)
사이즈,,
매우 작아요,31,2
조금 작아요,378,22
정사이즈예요,669,38
조금 커요,625,36
매우 커요,54,3


### 네 항목 모두 계산하기
4~5단계를 함수 하나로 묶어 모든 항목에 적용합니다.

In [31]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 4~5단계(표현 세기 → 답 고르기 → 비율 계산)를 함수로 묶고 모든 항목에 적용
# ------------------------------------------------------------

# name: 항목 이름 (예: "사이즈")
# 반환값: 행 = 5단계 답, 열 = '리뷰 수', '비율' 인 DataFrame
# 부수 효과: df에 항목 이름으로 된 열(리뷰별 답)을 추가하거나 덮어씀
def aspect_table(name):
    # 항목의 답 딕셔너리 가져오기
    levels = ASPECTS[name]
    # 그 항목의 표현만 세는 BoW 표 생성 (리뷰 × 표현)
    bow = count_phrases(levels)                                            # BoW로 표현 세기
    # 리뷰마다 답을 골라 df의 name 열에 저장
    df[name] = bow.apply(choose_level, axis=1, levels=levels)              # 리뷰별 답
    # 답별 리뷰 수를 사전 순서대로 정렬하고, 없는 답은 0으로 채움
    count = df[name].value_counts().reindex(list(levels), fill_value=0)    # 답별 리뷰 수
    # '리뷰 수'와 반올림한 정수 '비율'(%) 두 열을 가진 DataFrame 반환
    return pd.DataFrame({"리뷰 수": count, "비율": (count / count.sum() * 100).round().astype(int)})

# 딕셔너리 컴프리헨션
#   - ASPECTS의 항목 이름을 하나씩 꺼내 aspect_table(name)을 실행
#   - {항목 이름: 결과 표} 형태로 저장
results = {name: aspect_table(name) for name in ASPECTS}

# 항목별 결과 출력
for name, table in results.items():
    # table['리뷰 수'].sum(): 그 항목을 말한 리뷰 수 합계
    print(f"[{name}] 말한 리뷰 {table['리뷰 수'].sum()}건")
    # 표를 출력하고, 두 번째 인자 "\n"으로 표 뒤에 빈 줄 하나를 추가
    print(table, "\n")

[사이즈] 말한 리뷰 1757건
        리뷰 수  비율
사이즈             
매우 작아요    31   2
조금 작아요   378  22
정사이즈예요   669  38
조금 커요    625  36
매우 커요     54   3 

[화면 대비 색감] 말한 리뷰 110건
          리뷰 수  비율
화면 대비 색감          
매우 밝아요       5   5
약간 밝아요      60  55
화면과 비슷해요    21  19
약간 어두워요     24  22
매우 어두워요      0   0 

[퀄리티] 말한 리뷰 1095건
         리뷰 수  비율
퀄리티              
매우 별로예요    25   2
별로예요      302  28
괜찮아요       95   9
좋아요       628  57
매우 좋아요     45   4 

[보온성] 말한 리뷰 1041건
             리뷰 수  비율
보온성                  
전혀 따뜻하지 않아요    34   3
조금 따뜻해요         7   1
적당해요            6   1
꽤 따뜻해요        911  88
매우 따뜻해요        83   8 



---
## 6. 평가 화면 만들기
계산한 표를 **HTML**로 그립니다.

| 화면 요소 | 만드는 방법 |
|---|---|
| 왼쪽 항목 이름 | `ASPECTS`의 항목 이름 + 말한 리뷰 수 |
| 답 이름 | `ASPECTS`의 5단계 답 |
| 막대 길이 | 비율(%)을 막대의 너비(`width`)로 |
| 파란 막대·굵은 글씨 | 비율이 **가장 높은** 답 |

In [32]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 항목별 결과 표를 HTML 문자열로 만들고 노트북에 화면으로 표시
# -----------------------------------------------------------
# HTML: 문자열로 된 HTML을 노트북 출력 영역에 렌더링해 보여 주는 IPython 도구
from IPython.display import HTML

# 한 항목의 HTML을 만드는 함수
def aspect_html(name, table):
    top = table["비율"].idxmax()                                           # 비율이 가장 높은 답
    rows = ""
    for i, (label, r) in enumerate(table.iterrows()):
        style = "color:#111;font-weight:bold" if label == top else "color:#888"
        color = "#2a5bf6" if label == top else "#bbb"
        title = {0: name, 1: f'<small>({table["리뷰 수"].sum():,}건)</small>'}.get(i, "")   # 첫 줄 항목 이름, 둘째 줄 리뷰 수
        rows += f"""
        <div class="row">
          <div class="name">{title}</div>
          <div style="{style}">{label}</div>
          <div class="track"><div class="fill" style="width:{r['비율']}%;background:{color}"></div></div>
          <div class="pct" style="{style}">{r['비율']}%</div>
        </div>"""
    return f'<div class="group">{rows}</div>'

# 화면 모양(CSS)
style = """
<style>
.box   {font-family:sans-serif;background:#f4f4f4;padding:20px 24px;max-width:760px;border-radius:6px}
.head  {font-weight:bold;font-size:15px;margin-bottom:12px}
.group {border-top:1px solid #e2e2e2;padding:10px 0}
.group:first-of-type {border-top:none}
.row   {display:grid;grid-template-columns:110px 1fr 190px 50px;align-items:center;font-size:13px;height:22px}
.name  {color:#888;line-height:1.2}
.track {background:#ddd;height:6px;border-radius:3px;margin-right:12px}
.fill  {height:6px;border-radius:3px}
.pct   {text-align:right}
.note  {color:#999;font-size:11px;margin-top:8px}
</style>
"""

# 전체 화면 조립
groups = "".join(aspect_html(name, table) for name, table in results.items())
page = f"""{style}
<div class="box">
  <div class="head">리뷰 {len(df):,}건으로 분석한 평가예요</div>
  {groups}
  <div class="note">* 리뷰 글에 나온 표현(N-gram)을 세어 추정한 결과입니다. 항목을 말한 리뷰만 계산했습니다.</div>
</div>"""

# 노트북에 화면 보여 주기
HTML(page)

In [33]:
# 화면을 HTML 파일로 저장 (Colab 📁 파일 창에서 내려받아 브라우저로 열 수 있음)
with open("review_rating.html", "w", encoding="utf-8") as f:
    f.write("<meta charset='utf-8'>" + page)
print("저장 완료: review_rating.html")

저장 완료: review_rating.html


---
## 7. 정리

### ✅ 결과 확인
아래 셀에서 항목과 답을 바꿔 가며 **실제 리뷰가 그 답과 맞는지** 직접 읽어 보세요.

In [30]:
# ------------------------------------------------------------
# 이 셀에서 하는 일: 특정 항목에서 특정 답으로 분류된 리뷰 원문을 몇 개 출력
# ------------------------------------------------------------

# ===== ✏️ 확인할 항목과 답 =====
# check_name: ASPECTS의 항목 이름 중 하나 (aspect_table 실행으로 df에 같은 이름의 열이 있어야 함)
check_name = "사이즈"
# check_level: 그 항목의 5단계 답 이름 중 하나 (철자가 정확히 같아야 함)
check_level = "매우 커요"
# ===============================

# df[check_name] == check_level : 그 항목의 답이 check_level인 행만 True
# df.loc[조건, "상품평"]         : 조건이 True인 행의 '상품평' 열만 선택
# .head(5)                       : 위에서 5개만
# for text in ...                : 리뷰 원문을 하나씩 꺼내 '- ' 뒤에 붙여 출력
for text in df.loc[df[check_name] == check_level, "상품평"].head(5):
    print("-", text)

- 검정색 바지가 너무 커서 못 입을 것 같습니다. 사이즈 차이가 너무 나요
나머지 2개는 입기 편하고 맘에 듭니다.
- 아이가 키는큰데 너무말라서 옷고르기가 여간  힘든게 아닌데 역시나 젤 작은 사이즈를 샀는데도 길이는 딱인데 헐가 많이 크네요~
커서 벨트를 하고 입어야할듯요~
- 상품평처럼 검정바지가 너무너무 커요.ㅠㅠ
하나만 교환해주지도 않으실거면 첨부터 작은 사이즈로 넣어주시지ᆢ.
다른 색상은 신랑이 편하다고 맘에든다고 했는데 검정색 바지로 별하나 뺏어요.
이부분은 정말 고려해보시는게 어떨지요.
- 여자바지는 스키니로 너무 몸에 붙어서 남자바지로 구입했는데 일자바지여서 너무 크고 여유도 너무 있어서 벙벙해요
- 사이즈가 너무 큽니다. 
평소 31,32 입는데 32가 너무 커서 30으로 교환신청했어요.


### 오늘 쓴 기술 정리
| 단계 | 기술 | 도구 | 한 일 |
|---|---|---|---|
| 2 | N-gram + BoW | `CountVectorizer(ngram_range=(2, 2))` | 모든 2-gram을 세어 정도 표현 찾기 |
| 4 | N-gram + BoW | `CountVectorizer(vocabulary=..., ngram_range=(1, 3))` | 정해 둔 표현만 세어 리뷰별 답 고르기 |
| 4 | N-gram 길이 | `len(p.split())` | 표현이 여러 개면 긴 표현 우선 |

### 🤔 생각해 볼 점 (한계)
| 한계 | 예 | 더 나은 방법 |
|---|---|---|
| **설문이 아니라 추정** | 쇼핑몰 화면은 구매자가 고른 답, 오늘 결과는 글에서 추정한 답 | 구매자 설문 데이터 사용 |
| **"보통" 답이 적게 나옴** | 리뷰에 "보통이에요"라고 쓰는 사람은 적음 → 쇼핑몰처럼 가운데 답이 1등이 되기 어려움 | 표현 사전 보강 |
| **단어 뜻이 여러 개** | `밝다`가 "밝은 색을 샀어요"(색 고르기), `따뜻하다`가 "따뜻한 색감"일 수도 있음 | 항목 단어 주변만 보기, 문장 구조 분석 |
| **떨어진 부정 표현** | "따뜻해 보이지 **않아요**"는 `따뜻하다 보이다 않다`라 `따뜻하다 않다`에 안 걸림 | 3-gram 표현 추가 |
| **표현 사전은 사람이 만든 것** | `크다 주문`(크게 주문했다)을 "조금 작아요"로 넣은 것은 하나의 선택 | 여러 사람이 함께 정하고 검토 |
| **정답과 비교하지 않음** | 비율이 정확한지 모름 | 사람이 직접 분류한 리뷰와 비교 |

- 비율은 반올림했기 때문에 다섯 개를 더하면 100%가 아닐 수 있습니다.
- 막대의 %는 **그 항목을 말한 리뷰** 중의 비율입니다. 그래서 항목 이름 아래에 리뷰 수를 함께 보여 줍니다.



**참고:** [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) · [Kiwi (kiwipiepy)](https://github.com/bab2min/kiwipiepy)